This generates the "ready" csv files that are used to train the kmrf model
-- if new data is wanted, run "data_collection.ipynb" and "data_collection_us_traded.ipynb" first

In [77]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
import certifi
import json
from fredapi import Fred
import os
import ssl

# Custom packages
import derive_features as dd

import warnings
warnings.filterwarnings("ignore")

# Environment variables
import dotenv
dotenv.load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
FMP_API_KEY = os.getenv("FMP_API_KEY")

In [78]:
fmp_idx = pd.read_csv('data/inputs/fmp_index_list.csv')
fmp_comm = pd.read_csv('data/inputs/fmp_commodity_list.csv')

us_equity_symbol_names = {
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}

us_treasury_symbol_names = {
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
}

int_equity_symbol_names = {
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
}

macro_codes_dict = {
    # RATE BENCHMARKS
    'DFF': 'Federal Funds Effective Rate',
    'SOFR': 'Secured Overnight Financing Rate',
    # TREASURY RATES
    'DGS1MO': '1-Month Treasury Rate',
    'DGS3MO': '3-Month Treasury Rate', 
    'DGS6MO': '6-Month Treasury Rate',
    'DGS1': '1-Year Treasury Rate',
    'DGS2': '2-Year Treasury Rate',
    'DGS3': '3-Year Treasury Rate',
    'DGS5': '5-Year Treasury Rate',
    'DGS7': '7-Year Treasury Rate',
    'DGS10': '10-Year Treasury Rate',
    'DGS20': '20-Year Treasury Rate',
    'DGS30': '30-Year Treasury Rate',
    # OTHER RATES
    'DAAA': 'Moody\'s Seasoned AAA Corporate Bond Yield',
    'DBAA': 'Moody\'s Seasoned BAA Corporate Bond Yield',
    'OBMMCONF30YF': '30-Year Fixed Rate Conforming Mortgage Index',
    'DPRIME': 'Bank Prime Loan Rate',
    'T5YIE': '5-Year Breakeven Inflation Rate',
    'T10YIE': '10-Year Breakeven Inflation Rate',
    'T30YIE': '30-Year Breakeven Inflation Rate',
    # ECONOMIC INDICATORS
    'GDP': 'Gross Domestic Product',
    'GDPC1': 'Real Gross Domestic Product',
    'A939RX0Q048SBEA': 'Real GDP Per Capita',
    'PCE': 'Personal Consumption Expenditures',
    'PCEPI': 'Personal Consumption Expenditures Price Index',
    'PCEC96': 'Real Personal Consumption Expenditures',
    'CPIAUCSL': 'Consumer Price Index',
    'CPILFESL': 'Core CPI (Less Food and Energy)',
    'UNRATE': 'Unemployment Rate',
    'CIVPART': 'Labor Force Participation Rate',
    'INDPRO': 'Industrial Production Index',
    'PAYEMS': 'Total Nonfarm Payrolls',
    'HOUST': 'Housing Starts',
    'PERMIT': 'Building Permits',
    'MTSDS133FMS': 'Monthly US Government Surplus/Deficit',
    'GFDEGDQ188S': 'Federal Government Debt to GDP Ratio',
    'PMSAVE': 'Personal Savings',
    'PSAVERT': 'Personal Saving Rate',
    'GPDI': 'Gross Private Domestic Investment',
    'GPDIC1': 'Real Gross Private Domestic Investment',
    'BOGZ1FU263092001Q': 'Foreign Direct Investment in the United States',
    'QBPBSTAS': 'Balance Sheet - Total Assets',
    'FDHBFRBN': 'Federal Debt Held by Federal Reserve Banks',
    'FYGFDPUN': 'Federal Debt Held by the Public',
    'FDHBFIN': 'Federal Debt Held by Foreign Investors',
    'COMPOUT': 'Commercial Paper Outstanding',
    'ABCOMP': 'Asset-Backed Commercial Paper Outstanding',
    # MONEY SUPPLY
    'M1SL': 'M1 Money Stock',
    'M2SL': 'M2 Money Stock',
    'BASE': 'St. Louis Adjusted Monetary Base',
    # MARKET INDICATORS
    'VIXCLS': 'CBOE Volatility Index (VIX)',
    'UMCSENT': 'University of Michigan Consumer Sentiment',
    'USSLIND': 'Leading Index for the United States',
    'VISASMIHSA': 'Visa U.S. Consumer Spending Momentum Index: Headline',
    'VISASMIDSA': 'Visa U.S. Consumer Spending Momentum Index: Discretionary',
}

macro_units = {
    # INTEREST RATES AND FINANCIAL RATES
    'DFF': 'Percent, Seasonally Adjusted',
    'SOFR': 'Percent, Not Seasonally Adjusted',
    'DGS1MO': 'Percent, Not Seasonally Adjusted',
    'DGS3MO': 'Percent, Not Seasonally Adjusted',
    'DGS6MO': 'Percent, Not Seasonally Adjusted',
    'DGS1': 'Percent, Not Seasonally Adjusted',
    'DGS2': 'Percent, Not Seasonally Adjusted',
    'DGS3': 'Percent, Not Seasonally Adjusted',
    'DGS5': 'Percent, Not Seasonally Adjusted',
    'DGS7': 'Percent, Not Seasonally Adjusted',
    'DGS10': 'Percent, Not Seasonally Adjusted',
    'DGS20': 'Percent, Not Seasonally Adjusted',
    'DGS30': 'Percent, Not Seasonally Adjusted',
    'DAAA': 'Percent, Not Seasonally Adjusted',
    'DBAA': 'Percent, Not Seasonally Adjusted',
    'OBMMCONF30YF': 'Percent, Not Seasonally Adjusted',
    'DPRIME': 'Percent, Not Seasonally Adjusted',
    'T5YIE': 'Percent, Not Seasonally Adjusted',
    'T10YIE': 'Percent, Not Seasonally Adjusted',
    'T30YIE': 'Percent, Not Seasonally Adjusted',
    # ECONOMIC INDICATORS
    'GDP': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GDPC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'A939RX0Q048SBEA': 'Chained 2017 Dollars, Seasonally Adjusted',
    'PCE': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'PCEPI': 'Index 2017=100, Seasonally Adjusted',
    'PCEC96': 'Billions of Chained 2017 Dollars, Seasonally Adjusted',
    'CPIAUCSL': 'Index 1982-1984=100, Seasonally Adjusted',
    'CPILFESL': 'Index 1982-1984=100, Seasonally Adjusted',
    'UNRATE': 'Percent, Seasonally Adjusted',
    'CIVPART': 'Percent, Seasonally Adjusted',
    'INDPRO': 'Index 2017=100, Seasonally Adjusted',
    'PAYEMS': 'Thousands of Persons, Seasonally Adjusted',
    'HOUST': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'PERMIT': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'MTSDS133FMS': 'Millions of Dollars, Not Seasonally Adjusted',
    'GFDEGDQ188S': 'Percent of GDP, Not Seasonally Adjusted',
    'PMSAVE': 'Billions of Dollars, Not Seasonally Adjusted',
    'PSAVERT': 'Percent, Seasonally Adjusted',
    'GPDI': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GPDIC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'BOGZ1FU263092001Q': 'Millions of Dollars, Not Seasonally Adjusted',
    'QBPBSTAS': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFRBN': 'Billions of Dollars, Not Seasonally Adjusted',
    'FYGFDPUN': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFIN': 'Billions of Dollars, Not Seasonally Adjusted',
    'COMPOUT': 'Billions of Dollars, Not Seasonally Adjusted',
    'ABCOMP': 'Billions of Dollars, Not Seasonally Adjusted',
    # MONEY SUPPLY
    'M1SL': 'Billions of Dollars, Seasonally Adjusted',
    'M2SL': 'Billions of Dollars, Seasonally Adjusted',
    'BASE': 'Millions of Dollars, Not Seasonally Adjusted',
    # MARKET INDICATORS
    'VIXCLS': 'Index, Not Seasonally Adjusted',
    'UMCSENT': 'Index 1966:Q1=100, Not Seasonally Adjusted',
    'USSLIND': 'Percent, Seasonally Adjusted',
    'VISASMIHSA': 'Index, Seasonally Adjusted',
    'VISASMIDSA': 'Index, Seasonally Adjusted',
}


# US Equities

In [79]:
us_equity_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
us_equity_data.index = pd.to_datetime(us_equity_data.index)
us_equity_data.rename(columns=us_equity_symbol_names, level=0, inplace=True)

us_equity_always_disclude = ['Fidelity Nasdaq Composite Index ETF', 'Invesco S&P 500 Equal Weight ETF', 'Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
us_equity_disclude = [name for name in us_equity_data.columns.get_level_values(0) if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']

us_equity_include = list(set(us_equity_data.columns.get_level_values(0).tolist()) - set(us_equity_always_disclude)\
                          - set(us_equity_disclude) - set(int_equity_symbol_names.keys()) - set(us_treasury_symbol_names.keys()))
col_mask = us_equity_data.columns.map(lambda x: x[0] in us_equity_include)

us_equity_data = us_equity_data.loc[:, col_mask]

data_start_dates = {}
for asset in us_equity_include:
    data_start_dates[asset] = us_equity_data.xs(asset, level=0, axis=1)['close'].first_valid_index()

sorted(data_start_dates.items(), key=lambda x: x[1])

[('SPDR S&P 500 ETF', Timestamp('1993-01-29 00:00:00')),
 ('SPDR Dow Jones Industrial Average ETF', Timestamp('1998-01-20 00:00:00')),
 ('Financial Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Consumer Discretionary Select Sector SPDR',
  Timestamp('1998-12-22 00:00:00')),
 ('Energy Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Technology Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Materials Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Health Care Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Consumer Staples Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Industrial Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Utilities Select Sector SPDR', Timestamp('1998-12-22 00:00:00')),
 ('Invesco QQQ Trust', Timestamp('1999-03-10 00:00:00')),
 ('iShares S&P 500 Growth ETF', Timestamp('2000-05-26 00:00:00')),
 ('iShares Russell 2000 ETF', Timestamp('2000-05-26 00:00:00')),
 ('iShares S&P 500 Value ETF', 

In [80]:
us_equity_assets = us_equity_data.columns.get_level_values(0).unique().tolist()
us_equity_derived = dd.TimeSeriesDerivedFields(price_data=us_equity_data.xs(us_equity_assets[0], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=True)
us_equity_derived.columns = pd.MultiIndex.from_product([[us_equity_assets[0]], us_equity_derived.columns])
for i in range(1, len(us_equity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=us_equity_data.xs(us_equity_assets[i], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[us_equity_assets[i]], temp.columns])
    us_equity_derived = pd.concat([us_equity_derived, temp], axis=1)

us_equity_derived.to_csv('data/ready/us_equity.csv')

Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 8226 out of 8227
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 16452/16452 [00:02<00:00, 7612.20it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 6684 out of 6685
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 13368/13368 [00:01<00:00, 7732.07it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 6376 out of 6377
Extracting tsfresh features...


...

Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 6087 out of 6088
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 12174/12174 [00:01<00:00, 7960.64it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5065 out of 5066
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 10130/10130 [00:01<00:00, 6089.23it/s]


Generated 20 tsfresh features


# US Treasury ETFs

In [81]:
us_equity_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
us_equity_data.index = pd.to_datetime(us_equity_data.index)
us_equity_data.rename(columns=us_treasury_symbol_names, level=0, inplace=True)

col_mask = us_equity_data.columns.map(lambda x: x[0] in us_treasury_symbol_names.values())

us_treasury_data = us_equity_data.loc[:, col_mask]

In [82]:
us_treasury_assets = us_treasury_data.columns.get_level_values(0).unique().tolist()
us_treasury_derived = dd.TimeSeriesDerivedFields(price_data=us_treasury_data.xs(us_treasury_assets[0], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=True)
us_treasury_derived.columns = pd.MultiIndex.from_product([[us_treasury_assets[0]], us_treasury_derived.columns])
for i in range(1, len(us_treasury_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=us_treasury_data.xs(us_treasury_assets[i], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[us_treasury_assets[i]], temp.columns])
    us_treasury_derived = pd.concat([us_treasury_derived, temp], axis=1)

us_treasury_derived.to_csv('data/ready/us_treasury.csv')

Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 4617 out of 4618
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 9234/9234 [00:01<00:00, 6639.48it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5835 out of 5836
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11670/11670 [00:01<00:00, 6381.30it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5835 out of 5836
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 11670/11670 [00:01<00:00, 6414.18it/s]


Generated 20 tsfresh features


# International Equities

In [83]:
int_equity_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
int_equity_data.index = pd.to_datetime(int_equity_data.index)
int_equity_data.rename(columns=int_equity_symbol_names, level=0, inplace=True)

col_mask = int_equity_data.columns.map(lambda x: x[0] in int_equity_symbol_names.values())

int_equity_data = int_equity_data.loc[:, col_mask]

data_start_dates = {}
for asset in int_equity_symbol_names.values():
    data_start_dates[asset] = int_equity_data.xs(asset, level=0, axis=1)['close'].first_valid_index()

sorted(data_start_dates.items(), key=lambda x: x[1])

[('iShares MSCI Japan ETF', Timestamp('1996-03-18 00:00:00')),
 ('iShares China Large-Cap ETF', Timestamp('2004-10-08 00:00:00')),
 ('Vanguard FTSE Emerging Markets ETF', Timestamp('2005-03-10 00:00:00')),
 ('Vanguard FTSE Europe ETF', Timestamp('2005-03-10 00:00:00')),
 ('Vanguard FTSE Pacific ETF', Timestamp('2005-03-10 00:00:00')),
 ('Vanguard FTSE Developed Markets ETF', Timestamp('2007-07-26 00:00:00')),
 ('Vanguard Total International Stock ETF', Timestamp('2011-01-28 00:00:00')),
 ('iShares MSCI India ETF', Timestamp('2012-02-03 00:00:00'))]

In [84]:
int_equity_assets = int_equity_data.columns.get_level_values(0).unique().tolist()
int_equity_derived = dd.TimeSeriesDerivedFields(price_data=int_equity_data.xs(int_equity_assets[0], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=True)
int_equity_derived.columns = pd.MultiIndex.from_product([[int_equity_assets[0]], int_equity_derived.columns])
for i in range(1, len(int_equity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=int_equity_data.xs(int_equity_assets[i], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=True)
    temp.columns = pd.MultiIndex.from_product([[int_equity_assets[i]], temp.columns])
    int_equity_derived = pd.concat([int_equity_derived, temp], axis=1)

int_equity_derived.to_csv('data/ready/int_equity.csv')

Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 3692 out of 3693
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 7384/7384 [00:01<00:00, 7052.46it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 4577 out of 4578
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 9154/9154 [00:01<00:00, 6527.42it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 5175 out of 5176
Extracting tsfresh features...


...

Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 7435 out of 7436
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 14870/14870 [00:02<00:00, 7390.97it/s]


Generated 20 tsfresh features
Computing tsfresh features for columns: ['close_lag1d', 'volume_lag1d']
Window size: 20, Shift periods: [0]
Valid windows for feature extraction: 3436 out of 3437
Extracting tsfresh features...


Feature Extraction: 100%|██████████| 6872/6872 [00:00<00:00, 7898.83it/s]


Generated 20 tsfresh features


# Commodities

In [85]:
comm_symbol_name_dict = fmp_comm.set_index('symbol')['name'].to_dict()

commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0,1])
commodity_data.index = pd.to_datetime(commodity_data.index)

commodity_data.rename(columns=comm_symbol_name_dict, level=0, inplace=True)

commodity_data = commodity_data.loc[:, commodity_data.columns.map(lambda x: x[1] not in ['volume', 'vwap'])]
commodity_data = commodity_data.loc[:, commodity_data.columns.map(lambda x: x[0]!='Aluminum Futures')]
# Live Cattle Futures have data on non-tradin days
#   --> remove those days to maintain correct alignment for lagged data features
commodity_data = commodity_data.dropna(subset=pd.MultiIndex(levels=[['Gold Futures'], ['close']], codes=[[0], [0]]))

data_start_dates = {}
for asset in commodity_data.columns.get_level_values(0).unique():
    data_start_dates[asset] = commodity_data.xs(asset, level=0, axis=1)['close'].first_valid_index()

sorted(data_start_dates.items(), key=lambda x: x[1])

[('Gold Futures', Timestamp('1990-01-02 00:00:00')),
 ('Wheat Futures', Timestamp('1990-01-02 00:00:00')),
 ('Corn Futures', Timestamp('1990-01-02 00:00:00')),
 ('Copper', Timestamp('1990-01-02 00:00:00')),
 ('Sugar', Timestamp('1990-01-02 00:00:00')),
 ('Soybean Futures', Timestamp('1990-01-02 00:00:00')),
 ('Live Cattle Futures', Timestamp('1990-01-02 00:00:00')),
 ('Coffee', Timestamp('1990-01-02 00:00:00')),
 ('Natural Gas', Timestamp('2000-08-30 00:00:00')),
 ('Brent Crude Oil', Timestamp('2006-05-25 00:00:00'))]

In [86]:
commodity_assets = commodity_data.columns.get_level_values(0).unique().tolist()
commodity_equity_derived = dd.TimeSeriesDerivedFields(price_data=commodity_data.xs(commodity_assets[0], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=False)
commodity_equity_derived.columns = pd.MultiIndex.from_product([[commodity_assets[0]], commodity_equity_derived.columns])
for i in range(1, len(commodity_assets)):
    temp = dd.TimeSeriesDerivedFields(price_data=commodity_data.xs(commodity_assets[i], level=0, axis=1).dropna(how='all')).compute_all_derived_fields(include_tsfresh=False)
    temp.columns = pd.MultiIndex.from_product([[commodity_assets[i]], temp.columns])
    commodity_equity_derived = pd.concat([commodity_equity_derived, temp], axis=1)
    
commodity_equity_derived.to_csv('data/ready/commodity.csv')

# Macro Data

In [87]:
macro_data_daily = pd.read_csv('data/processed/macro_data_daily.csv', parse_dates=['date'], index_col='date')
# macro_data_monthly = pd.read_csv('data/processed/macro_data_monthly.csv', parse_dates=['date'], index_col='date')
# macro_data_quarterly = pd.read_csv('data/processed/macro_data_quarterly.csv', parse_dates=['date'], index_col='date')
macro_data_daily.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}') for col in macro_data_daily.columns])

# Drop rows where VIX is NaN to ensure alignment on only trading days
macro_data_daily = macro_data_daily.droplevel(level=1, axis=1)
macro_data_daily = macro_data_daily.dropna(subset=macro_data_daily.columns[-1])

# Forward fill to populate Bond Market holidays
macro_data_daily = macro_data_daily.ffill()

# macro_data_monthly.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, monthly') for col in macro_data_monthly.columns])
# macro_data_quarterly.columns = pd.MultiIndex.from_tuples([(f'{macro_codes_dict[col]} ({col})', f'{macro_units[col].split(',')[0]}, quarterly') for col in macro_data_quarterly.columns])

## Forward fill less frequently updated macro data
# macro_data_monthly_ffill = macro_data_monthly.reindex(macro_data_daily.index).ffill()
# macro_data_quarterly_ffill = macro_data_quarterly.reindex(macro_data_daily.index).ffill()

# macro_data_all = pd.concat([macro_data_daily, macro_data_monthly_ffill, macro_data_quarterly_ffill], axis=1)

# macro_data_all.to_csv('data/ready/macro_data_all.csv')

In [88]:
macro_data_daily_lagged = macro_data_daily.shift(1)
macro_data_daily_lagged.columns = [(f'{col} lag1d') for col in macro_data_daily_lagged.columns]
macro_data_daily_lagged['VIX 5d EMA lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].ewm(span=5, adjust=False).mean()
macro_data_daily_lagged['VIX 22d EMA lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].ewm(span=22, adjust=False).mean()
macro_data_daily_lagged['VIX 63d EMA lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].ewm(span=63, adjust=False).mean()
macro_data_daily_lagged['VIX 5d Std Dev lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].rolling(window=5).std()
macro_data_daily_lagged['VIX 22d Std Dev lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].rolling(window=22).std()
macro_data_daily_lagged['VIX 63d Std Dev lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].rolling(window=63).std()
macro_data_daily_lagged['VIX 1d Change lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].diff()
macro_data_daily_lagged['VIX 5d Change lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].diff(periods=5)
macro_data_daily_lagged['VIX 22d Change lag1d'] = macro_data_daily_lagged['CBOE Volatility Index (VIX) (VIXCLS) lag1d'].diff(periods=22)

macro_data_daily_lagged.iloc[1:].to_csv('data/ready/macro_data_daily.csv')

# 